In [1]:
"""
STR-BUIP Exchange Rate Model Estimation
========================================

This notebook reproduces the main results for:
- STR (Smooth Transition Regression) models
- BUIP (Behavioral UIP) models

All estimation functions are located in src/analysis.py for code organization.
"""

# Core imports
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Local imports
from src.config import config
from src.data_builder import DataBuilder
from src.analysis import build_z_candidates, estimate_all_countries

# Set publication-quality plotting style (black & white)
plt.rcParams.update({
    'figure.dpi': 300,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica'],
    'font.size': 11,
    'axes.edgecolor': 'black',
    'axes.facecolor': 'white',
    'axes.spines.top': True,
    'axes.spines.right': True,
    'axes.spines.left': True,
    'axes.spines.bottom': True,
    'axes.grid': False,
})

# Setup output directories
output_dir = config.OUTPUT_DIR
tables_dir = output_dir / 'tables'
figures_dir = output_dir / 'figures'
agg_fig_dir = figures_dir / 'aggregate'
country_fig_dir = figures_dir / 'country'

for d in [tables_dir, agg_fig_dir, country_fig_dir]:
    d.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("STR-BUIP ESTIMATION PIPELINE")
print("=" * 60)
print(f"\nOutput directories:")
print(f"  Tables: {tables_dir}")
print(f"  Aggregate Figures: {agg_fig_dir}")
print(f"  Country Figures: {country_fig_dir}")

STR-BUIP ESTIMATION PIPELINE

Output directories:
  Tables: /Users/lollo/Documents/Current_Projects/str-buip-estimation/results/tables
  Aggregate Figures: /Users/lollo/Documents/Current_Projects/str-buip-estimation/results/figures/aggregate
  Country Figures: /Users/lollo/Documents/Current_Projects/str-buip-estimation/results/figures/country


## 1. Load and Validate Data

In [2]:
"""
Load Panel Data
---------------
The dataset contains monthly exchange rate and macroeconomic data for 14 countries.
If the data file doesn't exist, it will be automatically built from DBnomics sources.
"""

data_path = config.DATA_DIR / "df_panel_final.csv"

if not data_path.exists():
    print("Building dataset from DBnomics...")
    builder = DataBuilder()
    df_panel = builder.run()
else:
    print(f"Loading existing dataset from {data_path}")
    df_panel = pd.read_csv(data_path, parse_dates=["date"])

# Define country list
COUNTRIES = {
    "Australia": "AUS",
    "Brazil": "BRA",
    "Canada": "CAN",
    "Euro Area": "EA",
    "Indonesia": "IND",
    "Japan": "JPN",
    "Korea": "KOR",
    "Mexico": "MEX",
    "New Zealand": "NZL",
    "Philippines": "PHL",
    "Switzerland": "CHE",
    "Thailand": "THA",
    "Türkiye": "TUR",
    "United Kingdom": "GBR"
}

print(f"\nDataset Summary:")
print(f"  Total observations: {len(df_panel):,}")
print(f"  Countries: {len(COUNTRIES)}")
print(f"  Date range: {df_panel['date'].min()} to {df_panel['date'].max()}")
print(f"\nObservations per country:")
print(df_panel['country'].value_counts().sort_index())

Building dataset from DBnomics...


Could not load series: {'dataset_code': 'IFS', 'message': 'Could not load series', 'provider_code': 'IMF', 'series_code': 'M.AU.PCPI_IX'}
Could not load series: {'dataset_code': 'IFS', 'message': 'Could not load series', 'provider_code': 'IMF', 'series_code': 'M.NZ.PCPI_IX'}
Could not load series: {'dataset_code': 'IFS', 'message': 'Could not load series', 'provider_code': 'IMF', 'series_code': 'M.NZ.PCPI_IX'}
Could not load series: {'dataset_code': 'IFS', 'message': 'Could not load series', 'provider_code': 'IMF', 'series_code': 'M.U2.PCPI_IX'}
Could not load series: {'dataset_code': 'IFS', 'message': 'Could not load series', 'provider_code': 'IMF', 'series_code': 'M.U2.PCPI_IX'}



Dataset Summary:
  Total observations: 4,186
  Countries: 14
  Date range: 2000-02-29 00:00:00 to 2024-12-31 00:00:00

Observations per country:
country
Australia         299
Brazil            299
Canada            299
Euro Area         299
Indonesia         299
Japan             299
Korea             299
Mexico            299
New Zealand       299
Philippines       299
Switzerland       299
Thailand          299
Türkiye           299
United Kingdom    299
Name: count, dtype: int64


## 2. Descriptive Statistics

Generate summary statistics for exchange rate returns and fundamental values across all countries.

In [3]:
"""
Compute descriptive statistics for:
- Exchange rate returns (r_s)
- Fundamental value (log CPI differential: log(p_dom) - log(p_for))
"""
from scipy import stats as sp_stats

stats_rows = []
for country_name in COUNTRIES.keys():
    sub = df_panel[df_panel['country'] == country_name].copy()
    if len(sub) == 0:
        continue
    
    # Exchange rate returns
    r_s = pd.to_numeric(sub['r_s'], errors='coerce').dropna()
    
    # Fundamental value (log CPI differential)
    fv = (np.log(sub['p_dom']) - np.log(sub['p_for'])).dropna()
    
    stats_rows.append({
        'Country': country_name,
        'Mean_r_s': r_s.mean(),
        'Var_r_s': r_s.var(ddof=1),
        'Skew_r_s': sp_stats.skew(r_s),
        'Kurt_r_s': sp_stats.kurtosis(r_s),
        'Mean_f': fv.mean(),
        'Var_f': fv.var(ddof=1)
    })

df_desc = pd.DataFrame(stats_rows).set_index('Country')
print("\n=== DESCRIPTIVE STATISTICS ===")
print(df_desc.round(4))

# Save outputs
csv_path = tables_dir / 'descriptive_statistics.csv'
tex_path = tables_dir / 'descriptive_statistics.tex'
df_desc.round(4).to_csv(csv_path)

# Create LaTeX version with math formatting
df_desc_latex = df_desc.copy()
df_desc_latex.columns = [
    r'$Mean_{\Delta s}$',
    r'$Var_{\Delta s}$',
    r'$Skew_{\Delta s}$',
    r'$Kurt_{\Delta s}$',
    r'$Mean_{f}$',
    r'$Var_{f}$'
]
with open(tex_path, 'w') as f:
    f.write(df_desc_latex.round(4).to_latex(
        float_format=lambda x: f"{x:.4f}", 
        escape=False, 
        caption='Descriptive Statistics: Exchange Rate Returns & Fundamental Value', 
        label='tab:desc_stats'
    ))

print(f"\n✓ Saved: {csv_path.name}")
print(f"✓ Saved: {tex_path.name}")


=== DESCRIPTIVE STATISTICS ===
                Mean_r_s  Var_r_s  Skew_r_s  Kurt_r_s  Mean_f   Var_f
Country                                                              
Australia         0.0001   0.0012    0.5630    2.0294  0.2773  0.1074
Brazil            0.0041   0.0023    0.8599    2.9949  1.0406  0.2732
Canada           -0.0000   0.0006    0.4183    2.5478 -0.2308  0.1242
Euro Area        -0.0002   0.0007    0.2075    1.3785 -0.0918  0.1642
Indonesia         0.0026   0.0010    0.0268    7.0811  1.2408  0.3364
Japan             0.0013   0.0007    0.1524    0.3847 -1.9950  0.5103
Korea             0.0009   0.0009    0.1386    5.1589  0.3488  0.4060
Mexico            0.0025   0.0011    1.3471    7.0182  0.6717  0.1294
New Zealand      -0.0004   0.0013    0.4077    0.5024  0.1383  0.2145
Philippines       0.0012   0.0003    0.8926    3.8278  0.7571  0.2318
Switzerland      -0.0020   0.0008   -0.1702    1.7484 -1.9071  1.4828
Thailand         -0.0003   0.0004    0.1461    0.9703  0.2

## 3. Model Estimation

Estimate STR and BUIP models for all countries.

**STR Model**: Smooth Transition Regression with automatic transition variable selection via linearity tests.

**BUIP Model**: Behavioral Uncovered Interest Parity with utility-based regime switching.

All estimation functions are in `src/analysis.py` for code organization.

In [4]:
"""
Estimate STR and BUIP models for all countries.

This will:
1. For STR: Test 35 transition variable candidates, select best via LM test, estimate via grid search + NLS
2. For BUIP: Estimate behavioral model with multistart optimization

Progress and results will be printed for each country.
"""

# Run estimation pipeline (defined in src/analysis.py)
str_results, buip_results = estimate_all_countries(df_panel, COUNTRIES, model_type='both')

print(f"\n{'='*60}")
print(f"ESTIMATION SUMMARY")
print(f"{'='*60}")
print(f"STR models successfully estimated: {len(str_results)} / {len(COUNTRIES)}")
print(f"BUIP models successfully estimated: {len(buip_results)} / {len(COUNTRIES)}")


Processing Australia
Running linearity tests for 35 candidates...
Best transition variable: eta_abs_lag4 (p=0.0035)
Running grid search...
Best transition variable: eta_abs_lag4 (p=0.0035)
Running grid search...
Grid search complete. Starting NLS...

STR Results for Australia:
const       0.003592
beta_c     -0.337267
beta_f      0.020466
gamma     561.100545
c           0.042039
const1     -0.003577
dtype: float64
AIC: -1991.67, BIC: -1969.55

BUIP: Australia
Sample size: 296
Running multistart optimization...
Grid search complete. Starting NLS...

STR Results for Australia:
const       0.003592
beta_c     -0.337267
beta_f      0.020466
gamma     561.100545
c           0.042039
const1     -0.003577
dtype: float64
AIC: -1991.67, BIC: -1969.55

BUIP: Australia
Sample size: 296
Running multistart optimization...

BUIP Results for Australia:
beta_f      0.020925
beta_c   -275.407867
gamma      48.236962
c         -31.413000
const     -67.955155
const1     67.958239
dtype: float64
AIC: -1

### 3.1 Transition Variable Selection Table

Shows which transition variable was selected for each country based on the LM linearity test.

In [10]:
"""
Generate table showing the selected transition variable for each country.
The transition variable with the lowest p-value in the LM linearity test is selected.
"""
print("\n" + "="*60)
print("TRANSITION VARIABLE SELECTION TABLE")
print("="*60)

# Build table from STR results
tv_selection_rows = []
for country_name in COUNTRIES.keys():
    if country_name in str_results:
        result_dict = str_results[country_name]
        tv_selection_rows.append({
            'Country': country_name,
            'Transition Variable': result_dict['z_name'],
            'P-value': f"{result_dict['lm_pval']:.4f}"
        })
    else:
        tv_selection_rows.append({
            'Country': country_name,
            'Transition Variable': 'N/A',
            'P-value': 'N/A'
        })

df_tv_selection = pd.DataFrame(tv_selection_rows)
print("\n")
print(df_tv_selection.to_string(index=False))

# Save outputs
csv_path = tables_dir / 'transition_variable_selection.csv'
tex_path = tables_dir / 'transition_variable_selection.tex'

df_tv_selection.to_csv(csv_path, index=False)
print(f"\n✓ Saved: {csv_path.name}")

# Create LaTeX version
with open(tex_path, 'w') as f:
    f.write(df_tv_selection.to_latex(
        index=False,
        escape=False,
        caption='Selected Transition Variables for STR Models (LM Linearity Test)',
        label='tab:transition_variables'
    ))
print(f"✓ Saved: {tex_path.name}")


TRANSITION VARIABLE SELECTION TABLE


       Country       Transition Variable P-value
     Australia              eta_abs_lag4  0.0035
        Brazil              eta_abs_lag2  0.0000
        Canada rel_misalignment_abs_lag2  0.0019
     Euro Area              ppp_abs_lag4  0.0000
     Indonesia                  eta_lag3  0.0000
         Japan                  eta_lag2  0.0328
         Korea rel_misalignment_abs_lag1  0.0000
        Mexico                  eta_lag2  0.0000
   New Zealand              drf_abs_lag2  0.0001
   Philippines              ppp_abs_lag1  0.0000
   Switzerland              drf_abs_lag2  0.0000
      Thailand rel_misalignment_abs_lag3  0.0049
       Türkiye                   ID_lag2  0.0000
United Kingdom              ppp_abs_lag5  0.0000

✓ Saved: transition_variable_selection.csv
✓ Saved: transition_variable_selection.tex


## 4. Results Tables

Generate publication-ready tables with parameter estimates and significance stars.

In [5]:
"""
Generate results tables for STR and BUIP models.
Estimates significant at 5% level are shown in bold.
"""

def format_with_stars(estimate, pval):
    """Format estimates with significance stars. Bold if p < 0.05."""
    if np.isnan(pval) or np.isnan(estimate):
        return ''
    val_str = f"{estimate:.4f}"
    if pval < 0.01:
        return f"\\textbf{{{val_str}}}***"
    if pval < 0.05:
        return f"\\textbf{{{val_str}}}**"
    if pval < 0.10:
        return f"{val_str}*"
    return val_str

# ============================================================================
# STR Results Table
# ============================================================================
print("\n" + "="*60)
print("GENERATING STR RESULTS TABLE")
print("="*60)

str_rows = []
for country_name in COUNTRIES.keys():
    if country_name in str_results:
        res = str_results[country_name]['result']
        row = {'Country': country_name}
        for param in ['const', 'const1', 'beta_c', 'beta_f', 'gamma', 'c']:
            est = res.params.get(param, np.nan)
            pval = res.pvals.get(param, np.nan)
            row[param] = format_with_stars(est, pval)
        str_rows.append(row)
    else:
        str_rows.append({
            'Country': country_name,
            'const': '', 'const1': '', 'beta_c': '', 'beta_f': '', 'gamma': '', 'c': ''
        })

df_str_table = pd.DataFrame(str_rows).set_index('Country')
print(df_str_table)

# Save outputs
str_csv = tables_dir / 'str_results_table.csv'
str_tex = tables_dir / 'str_results_table.tex'
df_str_table.to_csv(str_csv)
with open(str_tex, 'w') as f:
    f.write(df_str_table.to_latex(escape=False, caption='STR Model Estimates', label='tab:str_results'))

print(f"\n✓ Saved: {str_csv.name}")
print(f"✓ Saved: {str_tex.name}")

# ============================================================================
# BUIP Results Table
# ============================================================================
print("\n" + "="*60)
print("GENERATING BUIP RESULTS TABLE")
print("="*60)

buip_rows = []
for country_name in COUNTRIES.keys():
    if country_name in buip_results:
        res = buip_results[country_name]
        row = {'Country': country_name}
        for param in ['const', 'const1', 'beta_c', 'beta_f', 'gamma', 'c']:
            est = res.params.get(param, np.nan)
            pval = res.pvals.get(param, np.nan)
            row[param] = format_with_stars(est, pval)
        buip_rows.append(row)
    else:
        buip_rows.append({
            'Country': country_name,
            'const': '', 'const1': '', 'beta_c': '', 'beta_f': '', 'gamma': '', 'c': ''
        })

df_buip_table = pd.DataFrame(buip_rows).set_index('Country')
print(df_buip_table)

# Save outputs
buip_csv = tables_dir / 'buip_results_table.csv'
buip_tex = tables_dir / 'buip_results_table.tex'
df_buip_table.to_csv(buip_csv)
with open(buip_tex, 'w') as f:
    f.write(df_buip_table.to_latex(escape=False, caption='BUIP Model Estimates', label='tab:buip_results'))

print(f"\n✓ Saved: {buip_csv.name}")
print(f"✓ Saved: {buip_tex.name}")


GENERATING STR RESULTS TABLE
                             const               const1              beta_c  \
Country                                                                       
Australia                   0.0036              -0.0036  \textbf{-0.3373}**   
Brazil          \textbf{0.0135}***              -0.0088              0.0749   
Canada                     -0.0027               0.0041  \textbf{0.2633}***   
Euro Area        \textbf{0.0225}**  \textbf{-0.0263}***             -0.1561   
Indonesia       \textbf{0.0062}***    \textbf{0.1172}**              0.0913   
Japan                      -0.0009   \textbf{0.1061}***              0.0585   
Korea                       0.0018             166.5434              0.1225   
Mexico          \textbf{0.0088}***   \textbf{0.0395}***   \textbf{0.1611}**   
New Zealand                -0.0123             662.6912             -0.0263   
Philippines     \textbf{0.0026}***    \textbf{0.0286}**              0.0534   
Switzerland     \textb

In [6]:
"""
Generate aggregate figures across all countries:
- Regime distribution histograms
- Regime prevalence comparisons
"""
print("\n" + "="*60)
print("GENERATING AGGREGATE FIGURES")
print("="*60)

# ============================================================================
# STR Regime Distribution
# ============================================================================
all_G_values = []
for country_name, result_dict in str_results.items():
    G = result_dict['result'].G.dropna()
    all_G_values.extend(G.values)

if all_G_values:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(all_G_values, bins=50, color='black', edgecolor='black', linewidth=0.8)
    ax.axvline(0.5, color='red', linestyle='--', linewidth=1)
    ax.set_xlabel('Transition Function G(z)')
    ax.set_ylabel('Frequency')
    ax.set_title('STR Regime Distribution')
    plt.tight_layout()
    
    fig_path_pdf = agg_fig_dir / 'str_regime_distribution.pdf'
    fig_path_png = agg_fig_dir / 'str_regime_distribution.png'
    plt.savefig(fig_path_pdf, bbox_inches='tight')
    plt.savefig(fig_path_png, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"✓ Saved: {fig_path_pdf.name}")

# ============================================================================
# BUIP Regime Distribution
# ============================================================================
all_omega_values = []
for country_name, result in buip_results.items():
    omega = result.omega.dropna()
    all_omega_values.extend(omega.values)

if all_omega_values:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(all_omega_values, bins=50, color='black', edgecolor='black', linewidth=0.8)
    ax.axvline(0.5, color='red', linestyle='--', linewidth=1)
    ax.set_xlabel('Mixing Weight ω')
    ax.set_ylabel('Frequency')
    ax.set_title('BUIP Regime Distribution')
    plt.tight_layout()
    
    fig_path_pdf = agg_fig_dir / 'buip_regime_distribution.pdf'
    fig_path_png = agg_fig_dir / 'buip_regime_distribution.png'
    plt.savefig(fig_path_pdf, bbox_inches='tight')
    plt.savefig(fig_path_png, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"✓ Saved: {fig_path_pdf.name}")

# ============================================================================
# Regime Prevalence Comparison (STR vs BUIP)
# ============================================================================
str_chartist_pct = (np.array(all_G_values) > 0.5).mean() * 100 if all_G_values else 0
str_fund_pct = 100 - str_chartist_pct
buip_chartist_pct = (np.array(all_omega_values) < 0.5).mean() * 100 if all_omega_values else 0
buip_fund_pct = 100 - buip_chartist_pct

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, model, chartist, fund in [
    (axes[0], 'STR', str_chartist_pct, str_fund_pct),
    (axes[1], 'BUIP', buip_chartist_pct, buip_fund_pct)
]:
    ax.bar(['Chartist', 'Fundamentalist'], [chartist, fund], 
           color='black', edgecolor='black', linewidth=0.9)
    ax.set_title(f'{model} Regime Prevalence')
    ax.set_ylim(0, 100)
    ax.set_ylabel('Prevalence (%)') if ax is axes[0] else None

plt.tight_layout()
fig_path_pdf = agg_fig_dir / 'regime_prevalence_comparison.pdf'
fig_path_png = agg_fig_dir / 'regime_prevalence_comparison.png'
plt.savefig(fig_path_pdf, bbox_inches='tight')
plt.savefig(fig_path_png, dpi=300, bbox_inches='tight')
plt.close(fig)
print(f"✓ Saved: {fig_path_pdf.name}")


GENERATING AGGREGATE FIGURES
✓ Saved: str_regime_distribution.pdf
✓ Saved: buip_regime_distribution.pdf
✓ Saved: str_regime_distribution.pdf
✓ Saved: buip_regime_distribution.pdf
✓ Saved: regime_prevalence_comparison.pdf
✓ Saved: regime_prevalence_comparison.pdf


## 5. Detailed Country Plots

Generate detailed diagnostic plots for each country showing transition functions and regime evolution.

In [7]:
"""
Generate detailed country-specific plots for STR and BUIP models.
Each plot shows the transition variable, transition function, and regime evolution.
"""
import importlib
import src.output_figures
importlib.reload(src.output_figures)
from src.output_figures import plot_country_detailed_analysis
from src.utils import build_beh_sample

print("\n" + "="*60)
print("GENERATING DETAILED COUNTRY PLOTS")
print("="*60)

# ============================================================================
# STR Detailed Plots
# ============================================================================
print("\nSTR plots:")
for country_name in str_results.keys():
    try:
        result_dict = str_results[country_name]
        str_result = result_dict['result']
        z_name = result_dict['z_name']
        
        # Get country data and transition variable
        country_df = df_panel[df_panel['country'] == country_name].copy().set_index('date').sort_index()
        z_candidates = build_z_candidates(country_df)
        z_series = z_candidates[z_name]
        y = country_df['r_s'] - (country_df['i_for'] - country_df['i_dom'])
        
        # Generate and save plot
        save_path = country_fig_dir / f"{country_name.replace(' ', '_')}_str_detailed.pdf"
        plot_country_detailed_analysis(
            y=y,
            transition_var=z_series,
            transition_func=str_result.G,
            c_threshold=str_result.params['c'],
            country=country_name,
            var_name=z_name,
            model_type='STR',
            save_path=save_path
        )
        print(f"  ✓ {country_name}")
    except Exception as e:
        print(f"  ✗ {country_name}: {e}")

# ============================================================================
# BUIP Detailed Plots
# ============================================================================
print("\nBUIP plots:")
for country_name in buip_results.keys():
    try:
        buip_result = buip_results[country_name]
        
        # Get country data
        country_df = df_panel[df_panel['country'] == country_name].copy().set_index('date').sort_index()
        buip_df = build_beh_sample(country_df)
        y = buip_df['y']
        utility_diff = buip_result.U_f - buip_result.U_c
        
        # Generate and save plot
        save_path = country_fig_dir / f"{country_name.replace(' ', '_')}_buip_detailed.pdf"
        plot_country_detailed_analysis(
            y=y,
            transition_var=utility_diff,
            transition_func=buip_result.omega,
            c_threshold=buip_result.params['c'],
            country=country_name,
            var_name='Utility Differential',
            model_type='BUIP',
            save_path=save_path
        )
        print(f"  ✓ {country_name}")
    except Exception as e:
        print(f"  ✗ {country_name}: {e}")

print("\n✓ Detailed country plots completed")


GENERATING DETAILED COUNTRY PLOTS

STR plots:
  ✓ Australia
  ✓ Brazil
  ✓ Australia
  ✓ Brazil
  ✓ Canada
  ✓ Euro Area
  ✓ Canada
  ✓ Euro Area
  ✓ Indonesia
  ✓ Japan
  ✓ Indonesia
  ✓ Japan
  ✓ Korea
  ✓ Mexico
  ✓ Korea
  ✓ Mexico
  ✓ New Zealand
  ✓ New Zealand
  ✓ Philippines
  ✓ Switzerland
  ✓ Philippines
  ✓ Switzerland
  ✓ Thailand
  ✓ Thailand
  ✓ Türkiye
  ✓ United Kingdom

BUIP plots:
  ✓ Türkiye
  ✓ United Kingdom

BUIP plots:
  ✓ Australia
  ✓ Brazil
  ✓ Australia
  ✓ Brazil
  ✓ Canada
  ✓ Euro Area
  ✓ Canada
  ✓ Euro Area
  ✓ Indonesia
  ✓ Japan
  ✓ Indonesia
  ✓ Japan
  ✓ Korea
  ✓ Korea
  ✓ Mexico
  ✓ New Zealand
  ✓ Mexico
  ✓ New Zealand
  ✓ Philippines
  ✓ Switzerland
  ✓ Philippines
  ✓ Switzerland
  ✓ Thailand
  ✓ Türkiye
  ✓ Thailand
  ✓ Türkiye
  ✓ United Kingdom

✓ Detailed country plots completed
  ✓ United Kingdom

✓ Detailed country plots completed


## 6. Model Comparison Figures

Compare gamma parameters and RMSE across countries for STR and BUIP models.

In [8]:
"""
Generate comparison figures:
- Gamma parameters by country
- RMSE comparison between models
"""
print("\n" + "="*60)
print("GENERATING COMPARISON FIGURES")
print("="*60)

# ============================================================================
# Gamma Comparison - STR
# ============================================================================
str_gamma = {c: res['result'].params.get('gamma', np.nan) for c, res in str_results.items()}

if str_gamma:
    items = sorted(str_gamma.items(), key=lambda x: x[1])
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar([i[0] for i in items], [i[1] for i in items], 
           color='black', edgecolor='black', linewidth=0.9)
    ax.set_ylabel('Gamma')
    ax.set_title('STR Gamma by Country')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    
    path_pdf = agg_fig_dir / 'str_gamma_comparison.pdf'
    path_png = agg_fig_dir / 'str_gamma_comparison.png'
    plt.savefig(path_pdf, bbox_inches='tight')
    plt.savefig(path_png, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"✓ Saved: {path_pdf.name}")

# ============================================================================
# Gamma Comparison - BUIP
# ============================================================================
buip_gamma = {c: res.params.get('gamma', np.nan) for c, res in buip_results.items()}

if buip_gamma:
    items = sorted(buip_gamma.items(), key=lambda x: x[1])
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar([i[0] for i in items], [i[1] for i in items], 
           color='black', edgecolor='black', linewidth=0.9)
    ax.set_ylabel('Gamma')
    ax.set_title('BUIP Gamma by Country')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    
    path_pdf = agg_fig_dir / 'buip_gamma_comparison.pdf'
    path_png = agg_fig_dir / 'buip_gamma_comparison.png'
    plt.savefig(path_pdf, bbox_inches='tight')
    plt.savefig(path_png, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"✓ Saved: {path_pdf.name}")

# ============================================================================
# RMSE Model Comparison (STR vs BUIP)
# ============================================================================
comparison_rows = []
for c in COUNTRIES.keys():
    str_res = str_results.get(c, {}).get('result', None)
    str_rmse_val = str_res.rmse if str_res else np.nan
    
    buip_res = buip_results.get(c, None)
    buip_rmse_val = buip_res.rmse if buip_res else np.nan
    
    if not np.isnan(str_rmse_val) or not np.isnan(buip_rmse_val):
        comparison_rows.append({
            'Country': c,
            'STR_RMSE': str_rmse_val,
            'BUIP_RMSE': buip_rmse_val
        })

if comparison_rows:
    df_rmse = pd.DataFrame(comparison_rows).set_index('Country')
    
    fig, ax = plt.subplots(figsize=(9, 4.5))
    x = np.arange(len(df_rmse.index))
    w = 0.35
    ax.bar(x - w/2, df_rmse['STR_RMSE'], width=w, 
           color='black', edgecolor='black', label='STR')
    ax.bar(x + w/2, df_rmse['BUIP_RMSE'], width=w, 
           color='darkgray', edgecolor='black', label='BUIP')
    ax.set_xticks(x)
    ax.set_xticklabels(df_rmse.index, rotation=45)
    ax.set_ylabel('RMSE')
    ax.set_title('Model Comparison (RMSE)')
    ax.legend(frameon=False)
    plt.tight_layout()
    
    path_pdf = agg_fig_dir / 'str_buip_rmse_comparison.pdf'
    path_png = agg_fig_dir / 'str_buip_rmse_comparison.png'
    plt.savefig(path_pdf, bbox_inches='tight')
    plt.savefig(path_png, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"✓ Saved: {path_pdf.name}")

print("\n✓ Comparison figures completed")


GENERATING COMPARISON FIGURES
✓ Saved: str_gamma_comparison.pdf
✓ Saved: str_gamma_comparison.pdf
✓ Saved: buip_gamma_comparison.pdf
✓ Saved: buip_gamma_comparison.pdf
✓ Saved: str_buip_rmse_comparison.pdf

✓ Comparison figures completed
✓ Saved: str_buip_rmse_comparison.pdf

✓ Comparison figures completed


## 7. Summary

Final summary of all generated outputs.

In [ ]:
"""
Summary of Pipeline Outputs
"""
print("\n" + "="*80)
print("REPRODUCTION PIPELINE COMPLETE")
print("="*80)

print(f"\nModels Estimated:")
print(f"  STR:  {len(str_results):2d} / {len(COUNTRIES)} countries")
print(f"  BUIP: {len(buip_results):2d} / {len(COUNTRIES)} countries")

print(f"\nTables Generated (CSV + LaTeX):")
tables = ['descriptive_statistics', 'transition_variable_selection', 'str_results_table', 'buip_results_table']
for t in tables:
    print(f"  • {t}")

print(f"\nAggregate Figures Generated (PDF + PNG):")
agg_figs = [
    'str_regime_distribution',
    'buip_regime_distribution',
    'regime_prevalence_comparison',
    'str_gamma_comparison',
    'buip_gamma_comparison',
    'str_buip_rmse_comparison'
]
for f in agg_figs:
    print(f"  • {f}")

print(f"\nCountry-Specific Plots:")
str_plots = len([f for f in country_fig_dir.glob('*_str_detailed.pdf')])
buip_plots = len([f for f in country_fig_dir.glob('*_buip_detailed.pdf')])
print(f"  • STR detailed plots:  {str_plots}")
print(f"  • BUIP detailed plots: {buip_plots}")

print(f"\nOutput Directories:")
print(f"  • Tables:            {tables_dir}")
print(f"  • Aggregate Figures: {agg_fig_dir}")
print(f"  • Country Figures:   {country_fig_dir}")

print("\n" + "="*80)
print("All results saved successfully!")
print("="*80)


REPRODUCTION PIPELINE COMPLETE

Models Estimated:
  STR:  14 / 14 countries
  BUIP: 14 / 14 countries

Tables Generated (CSV + LaTeX):
  • descriptive_statistics
  • str_results_table
  • buip_results_table

Aggregate Figures Generated (PDF + PNG):
  • str_regime_distribution
  • buip_regime_distribution
  • regime_prevalence_comparison
  • str_gamma_comparison
  • buip_gamma_comparison
  • str_buip_rmse_comparison

Country-Specific Plots:
  • STR detailed plots:  14
  • BUIP detailed plots: 14

Output Directories:
  • Tables:            /Users/lollo/Documents/Current_Projects/str-buip-estimation/results/tables
  • Aggregate Figures: /Users/lollo/Documents/Current_Projects/str-buip-estimation/results/figures/aggregate
  • Country Figures:   /Users/lollo/Documents/Current_Projects/str-buip-estimation/results/figures/country

All results saved successfully!
